In [ ]:
from openai import OpenAI
import pandas as pd
from radprompter import Prompt
import random
import re
from tqdm import tqdm

In [ ]:
import os
client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com/v1")
MODEL_NAME = "deepseek-chat"


In [ ]:
prompt = Prompt("../data_prep_config/prompt-anthropic.toml")
system_prompt = prompt.system_prompt
turn_1_user_prompt = prompt.user_prompts[0]
turn_1_response_template = prompt.response_templates[0].strip()

In [ ]:
def generate_random_features(num_features):
    all_features = [
        "name", "age", "date", "hospital", "address", "phone", "email", "id", "url", "device"
    ]
    
    # Always include "name" as it's essential for a medical report
    features = ["name"]
    
    # Randomly select the remaining features
    remaining_features = random.sample(all_features[1:], min(num_features - 1, len(all_features) - 1))
    features.extend(remaining_features)
    
    # Adjust the probability of selecting "long" based on the number of features
    length_probabilities = {
        "short": 0.1,
        "medium": 0.3,
        "long": 0.6
    }
    if num_features > 5:  # If there are more than 5 features, increase the chance of selecting "long"
        length_probabilities["long"] = 0.6
        length_probabilities["short"] = 0.2
        length_probabilities["medium"] = 0.2
    elif num_features > 8:  # If there are more than 8 features, increase the chance of selecting "long" even more
        length_probabilities["long"] = 0.8
        length_probabilities["short"] = 0.1
        length_probabilities["medium"] = 0.1
    
    # Select the report length based on the adjusted probabilities
    length = random.choices(["short", "medium", "long"], weights=[length_probabilities["short"], length_probabilities["medium"], length_probabilities["long"]], k=1)[0]
    
    return {
        "report_length": length,
        "features_to_include": ", ".join(features)
    }

# Example usage
def generate_random_inputs(num_inputs):
    inputs = []
    for _ in range(num_inputs):
        num_features = random.randint(5, 11)  # Minimum 2 features, maximum 11
        inputs.append(generate_random_features(num_features))
    return inputs

In [ ]:
num_inputs = 10000
criteria = generate_random_inputs(num_inputs)
records = {}
for i in tqdm(range(num_inputs)):
    criterion = criteria[i]
    message_1_variables = turn_1_user_prompt.replace("{{report_length}}", criterion["report_length"]).replace("{{features_to_include}}", criterion["features_to_include"])
    request = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message_1_variables},
        ],
        stream=False
    )
    response = request.choices[0].message.content
    raw_report = re.search(r"([\s\S]*?)</raw_report>", response).group(1)
    anonymized_report = re.search(r"<anonymized_report>([\s\S]*?)</anonymized_report>", response).group(1)

    records[f"message_{i}"] = {
        "criterion": criterion,
        "raw_report": raw_report,
        "anonymized_report": anonymized_report
    }

In [ ]:
# Save to CSV
df = pd.DataFrame.from_dict(records, orient="index")
df.to_csv("../data/deepseek/output_3_deepseek.csv", index=False)